In [ ]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.2"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!git clone https://github.com/jameswu05/searchless_chess_ml_opt.git
%cd searchless_chess_ml_opt

fatal: destination path 'searchless_chess_ml_opt' already exists and is not an empty directory.
/content/searchless_chess_ml_opt


In [ ]:
!pip install -U pip
!pip install -U "jax[cuda13]" orbax-checkpoint dm-haiku chex absl-py python-chess pandas scipy apache-beam[tfrecord]

In [ ]:
!pip install -r requirements.txt

In [ ]:
os.makedirs('data/train', exist_ok=True)
os.makedirs('data/test', exist_ok=True)

!wget -q https://storage.googleapis.com/searchless_chess/data/puzzles.csv -P data/
!wget -q https://storage.googleapis.com/searchless_chess/data/test/action_value_data.bag -P data/test/
!wget -q "https://storage.googleapis.com/searchless_chess/data/train/action_value-00000-of-02148_data.bag" -P data/train/
!mv data/train/action_value-00000-of-02148_data.bag data/train/action_value_data.bag
print("Done.")

Done.


In [ ]:
%cd ..

/content


In [ ]:
from pathlib import Path
import csv
import json
import sys

REPO_ROOT = Path("/content/searchless_chess_ml_opt")
SRC_DIR = REPO_ROOT / "src"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import orbax.checkpoint as ocp
import pandas as pd

import metrics_evaluator
import config as config_lib
import constants
import transformer
from tokenizer import SEQUENCE_LENGTH
import utils

def build_predictor(predictor_config):
    return transformer.build_transformer_predictor(config=predictor_config)

def build_eval_config(
    *,
    split: str,
    batch_size: int,
    num_eval_data: int | None,
    num_return_buckets: int,
    policy: str,
):
    return config_lib.EvalConfig(
        policy=policy,
        num_return_buckets=num_return_buckets,
        num_eval_data=num_eval_data,
        batch_size=batch_size,
        data=config_lib.DataConfig(
            batch_size=batch_size,
            shuffle=False,
            seed=0,
            drop_remainder=False,
            worker_count=0,
            num_return_buckets=num_return_buckets,
            split=split,
            policy=policy,
            num_records=None,
        ),
    )


def find_step_dir(checkpoints_root: Path, policy: str, step: int) -> Path:
    candidates = [
        checkpoints_root / policy / str(step),
        checkpoints_root / str(step),
    ]
    for c in candidates:
        if c.exists():
            return c
    raise FileNotFoundError(
        f"Could not find checkpoint step {step}. Tried: "
        + ", ".join(str(x) for x in candidates)
    )


def restore_params(step_dir: Path, tree_name: str, predictor):
    target_dir = step_dir / tree_name
    if not target_dir.exists():
        raise FileNotFoundError(f"Missing checkpoint tree: {target_dir}")

    import jax
    import numpy as np
    import orbax.checkpoint as ocp

    # Build an abstract tree with the exact parameter structure expected.
    dummy_params = predictor.initial_params(
        jax.random.PRNGKey(0),
        np.zeros((1, 79), dtype=np.uint32),
    )

    abstract_target = jax.tree.map(
        lambda x: ocp.ArrayRestoreArgs(
            restore_type=np.ndarray,
            global_shape=x.shape,
            dtype=x.dtype,
        ),
        dummy_params,
    )

    checkpointer = ocp.PyTreeCheckpointer()
    restored = checkpointer.restore(
        str(target_dir),
        item=abstract_target,
    )
    return restored


def evaluate_one_checkpoint(
    *,
    checkpoints_root: Path,
    policy: str,
    step: int,
    tree_name: str,
    split: str,
    batch_size: int,
    num_eval_data: int | None,
    num_return_buckets: int,
    predictor,
):
    step_dir = find_step_dir(checkpoints_root, policy, step)
    params = restore_params(step_dir, tree_name, predictor)

    eval_cfg = build_eval_config(
        split=split,
        batch_size=batch_size,
        num_eval_data=num_eval_data,
        num_return_buckets=num_return_buckets,
        policy=policy,
    )

    evaluator = metrics_evaluator.build_evaluator(
        predictor=predictor,
        config=eval_cfg,
    )
    metrics = evaluator.step(params=params, step=step)

    row = {"step": step, "tree": tree_name}
    for k, v in metrics.items():
        row[k] = maybe_floatify(v)
    return row

In [ ]:
CHECKPOINTS_ROOT = Path("/content/drive/MyDrive/chess_vae_checkpoints/action_value")
OUTPUT_CSV = REPO_ROOT / "selected_eval_results.csv"
STEPS = [50000, 100000, 500000, 1000000]

In [ ]:
!grep -R "action_value" /content/searchless_chess_ml_opt/src -n
!grep -R "Predictor" /content/searchless_chess_ml_opt/src -n
!grep -R "predictor =" /content/searchless_chess_ml_opt/src -n
!grep -R "constants.Predictor" /content/searchless_chess_ml_opt/src -n

/content/searchless_chess_ml_opt/src/constants.py:98:CODERS['action_value'] = coders.TupleCoder((
/content/searchless_chess_ml_opt/src/config.py:22:PolicyType = Literal['action_value', 'state_value', 'behavioral_cloning']
/content/searchless_chess_ml_opt/src/config.py:23:POLICY_TYPES = ['action_value', 'state_value', 'behavioral_cloning']
/content/searchless_chess_ml_opt/src/train.py:34:    'action_value',
/content/searchless_chess_ml_opt/src/train.py:47:    case 'action_value':
/content/searchless_chess_ml_opt/src/searchless_chess.ipynb:45:        "policy = 'action_value'\n",
/content/searchless_chess_ml_opt/src/searchless_chess.ipynb:49:        "  case 'action_value':\n",
/content/searchless_chess_ml_opt/src/engines/constants.py:44:      policy = 'action_value'
/content/searchless_chess_ml_opt/src/engines/constants.py:49:      policy = 'action_value'
/content/searchless_chess_ml_opt/src/engines/constants.py:54:      policy = 'action_value'
/content/searchless_chess_ml_opt/src/engines

In [ ]:
predictor_config = transformer.TransformerConfig(
    vocab_size=utils.NUM_ACTIONS,
    output_size=128,
    pos_encodings=transformer.PositionalEncodings.LEARNED,
    max_sequence_length=SEQUENCE_LENGTH + 2,
    num_heads=8,
    num_layers=8,
    embedding_dim=256,
    apply_post_ln=True,
    apply_qk_layernorm=False,
    use_causal_mask=False,
    latent_tokens=16,
    latent_dim=128,
    latent_decoder_layers=2,
    seed=1,
)

In [ ]:
step_dir = find_step_dir(CHECKPOINTS_ROOT, "action_value", 50000)
predictor = build_predictor(predictor_config)
params = restore_params(step_dir, "params_ema", predictor)
print("restored params")

restored params


In [ ]:
!find /content -name "action_value_data.bag"

find: ‘/content/drive/.Encrypted/.shortcut-targets-by-id/1sSB2Rmhlk4nDcq7oy1sErBWfeiImsEJN/Basketball Tournament’: No such file or directory
find: ‘/content/drive/.Encrypted/.shortcut-targets-by-id/1_RKe9MD7R2a6rmj0MV7Xl-0ge6yAPugI/RPI WBB 2023-24’: No such file or directory
find: ‘/content/drive/.Encrypted/.shortcut-targets-by-id/1dc5MKsGci5HDTCBc81NK6JJSgbAYnyPZ/SummerGame2025’: No such file or directory
/content/searchless_chess_ml_opt/data/train/action_value_data.bag
/content/searchless_chess_ml_opt/data/test/action_value_data.bag


In [ ]:
os.chdir("/content/searchless_chess_ml_opt/src")
print("cwd:", os.getcwd())

cwd: /content/searchless_chess_ml_opt/src


In [ ]:
import pandas as pd
import json

OUTPUT_CSV = "/content/searchless_chess_ml_opt/selected_eval_results.csv"

results = []

print("Repo root:", REPO_ROOT)
print("Checkpoints root:", CHECKPOINTS_ROOT)
print("Exists:", CHECKPOINTS_ROOT.exists())

predictor = build_predictor(predictor_config)

pd.DataFrame().to_csv(OUTPUT_CSV, index=False)

for step in STEPS:
    print(f"\n=== Evaluating step {step} ===")

    try:
        row = evaluate_one_checkpoint(
            checkpoints_root=CHECKPOINTS_ROOT,
            policy="action_value",
            step=step,
            tree_name="params_ema",
            split="test",
            batch_size=1,
            num_eval_data=32,
            num_return_buckets=128,
            predictor=predictor,
        )

        results.append(row)

        print(json.dumps(row, indent=2, sort_keys=True))

        df_row = pd.DataFrame([row])
        df_row.to_csv(
            OUTPUT_CSV,
            mode="a",
            header=not pd.io.common.file_exists(OUTPUT_CSV) or len(results) == 1,
            index=False,
        )

        print(f"Saved step {step} to CSV")

    except Exception as e:
        print(f"Failed at step {step}: {e}")

print("\nFinished all steps!")

df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False)

print("Final CSV saved to:", OUTPUT_CSV)
print(df)

Repo root: /content/searchless_chess_ml_opt
Checkpoints root: /content/drive/MyDrive/chess_vae_checkpoints/action_value
Exists: True

=== Evaluating step 50000 ===
{
  "eval_action_accuracy": 0.0625,
  "eval_entropy": 1.3206341544786296e-06,
  "eval_kendall_tau": -0.020227146218686404,
  "eval_l2_win_prob_loss": 0.17382592230247246,
  "eval_output_log_loss": 30.972634990531034,
  "step": 50000,
  "tree": "params_ema"
}
Saved step 50000 to CSV

=== Evaluating step 100000 ===
{
  "eval_action_accuracy": 0.03125,
  "eval_entropy": 3.8371217630351806e-25,
  "eval_kendall_tau": 0.04546015956741692,
  "eval_l2_win_prob_loss": 0.17382593242976285,
  "eval_output_log_loss": 506.0964269357424,
  "step": 100000,
  "tree": "params_ema"
}
Saved step 100000 to CSV

=== Evaluating step 500000 ===
{
  "eval_action_accuracy": 0.03125,
  "eval_entropy": 0.0,
  "eval_kendall_tau": 0.04546015956741692,
  "eval_l2_win_prob_loss": 0.17382593242976285,
  "eval_output_log_loss": 710.8074144088134,
  "step": 

In [ ]:
!cp /content/searchless_chess_ml_opt/selected_eval_results.csv /content/drive/MyDrive/